In [1]:
# !pip install pycaret

In [2]:
from pycaret.regression import *

# Loading Data

In [3]:
import pandas as pd
import numpy as np

In [4]:
# USE WHEN RUNNING ON COLAB
"""import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/data-dump")

print("Path to dataset files:", path)

df_bat = pd.read_csv(path + '/ipl_bat_features.csv')
df_bowl = pd.read_csv(path + '/ipl_bowl_features.csv')
df_bat_fp = pd.read_csv(path + '/ipl_bat_fp.csv')
df_bowl_fp = pd.read_csv(path + '/ipl_bowl_fp.csv')
"""

'import kagglehub\n\n# Download latest version\npath = kagglehub.dataset_download("s1nghayush/data-dump")\n\nprint("Path to dataset files:", path)\n\ndf_bat = pd.read_csv(path + \'/ipl_bat_features.csv\')\ndf_bowl = pd.read_csv(path + \'/ipl_bowl_features.csv\')\ndf_bat_fp = pd.read_csv(path + \'/ipl_bat_fp.csv\')\ndf_bowl_fp = pd.read_csv(path + \'/ipl_bowl_fp.csv\')\n'

In [5]:
# USE THIS WHEN RUNNING LOCALLY

df_bat = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\processed_data\ipl_bat_features.csv")
df_bowl = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\processed_data\ipl_bowl_features.csv")
df_bat_fp = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\processed_data\ipl_bat_fp.csv")
df_bowl_fp = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\processed_data\ipl_bowl_fp.csv")

In [6]:
df_bat.shape, df_bowl.shape

((24333, 34), (24444, 34))

# Data Preparation

In [7]:
df_bowl['date'] = pd.to_datetime(df_bowl['date'])
df_bat['date'] = pd.to_datetime(df_bat['date'])

In [8]:
# merging fantasy points in one single dataset

df_bat = df_bat.merge(
    df_bat_fp[['batting_fp', 'match_id', 'batter_id']],
    on = ['match_id', 'batter_id']
)

df_bowl = df_bowl.merge(
    df_bowl_fp[['bowling_fp', 'match_id', 'bowler_id']],
    on = ['match_id', 'bowler_id']
)

In [9]:
#sorting values by date and innings
df_bat = df_bat.sort_values(by = ['date', 'inns'])
df_bowl = df_bowl.sort_values(by = ['date', 'inns'])
df_bat.head()

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,venue,career_avg,career_rpm,...,ssn_rpm,ssn_sr,ssn_bpm,ssn_bfpm,last5_avg,last5_sr,last5_rpm,last5_bfpm,last5_bpm,batting_fp
929,2008-04-18,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,26184,Ajit Agarkar,M Chinnaswamy Stadium,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2340,2008-04-18,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,227712,Ashok Dinda,M Chinnaswamy Stadium,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3295,2008-04-18,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,37737,Brendon McCullum,M Chinnaswamy Stadium,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,298
4336,2008-04-18,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,5766,David Hussey,M Chinnaswamy Stadium,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16
7649,2008-04-18,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,236779,Ishant Sharma,M Chinnaswamy Stadium,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [10]:
df_bowl.head()

,date,match_id,bowl_team,bat_team,bowler_id,bowler,inns,venue,season,career_avg,...,ssn_eco,ssn_sr,ssn_wpm,ssn_opm,last5_avg,last5_eco,last5_sr,last5_wpm,last5_opm,bowling_fp
2384,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,6938,Ashley Noffke,1,M Chinnaswamy Stadium,2008,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31
2811,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,26206,Balachandra Akhil,1,M Chinnaswamy Stadium,2008,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3548,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,8291,Cameron White,1,M Chinnaswamy Stadium,2008,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
7924,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,45789,Jacques Kallis,1,M Chinnaswamy Stadium,2008,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29
12047,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,44111,Mark Boucher,1,M Chinnaswamy Stadium,2008,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [11]:
# Label Encoding categorical features

from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
for column in ['bat_team', 'bowl_team', 'venue']:
    df_bat[column] = encoder.fit_transform(df_bat[column])
for column in ['bat_team', 'bowl_team', 'venue']:
    df_bowl[column] = encoder.fit_transform(df_bowl[column])

In [12]:
# dropping columns not required for training
df_bat.drop(columns=['date', 'match_id', 'batter'], inplace=True)
df_bowl.drop(columns=['date', 'match_id', 'bowler'], inplace=True)

In [13]:
df_bat.shape, df_bowl.shape

((24333, 32), (24664, 32))

# Train Test Val Split

- For full dataset (2008-24), use 23 as val set, and 24 as test set

In [14]:
# TRAIN VAL TEST SPLIT

df_bat_train = df_bat[df_bat['season'] < 2023]
df_bat_val = df_bat[df_bat['season'] >= 2023]
df_bowl_train = df_bowl[df_bowl['season'] < 2023]
df_bowl_val = df_bowl[df_bowl['season'] >= 2023]

In [15]:
df_bat_train.shape, df_bat_val.shape,df_bowl_train.shape, df_bowl_val.shape

((20878, 32), (3455, 32), (21208, 32), (3456, 32))

# PyCaret Batting Model Training

In [16]:
reg_setup = setup(data=df_bat_train,
                  target='batting_fp',
                  train_size=0.99,
                  test_data = df_bat_val,
                  data_split_shuffle=False,
                  numeric_imputation=0,
                  session_id = 42)

,Description,Value
0,Session id,42
1,Target,batting_fp
2,Target type,Regression
3,Original data shape,"(24333, 32)"
4,Transformed data shape,"(24333, 32)"
5,Transformed train set shape,"(20878, 32)"
6,Transformed test set shape,"(3455, 32)"
7,Numeric features,31
8,Rows with missing values,35.3%
9,Preprocess,True


In [17]:
best_model = compare_models()

,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,01:41:35
Status,. . . . . . . . . . . . . . . . . .,Loading Dependencies
Estimator,. . . . . . . . . . . . . . . . . .,Compiling Library


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
gbr,Gradient Boosting Regressor,20.9897,1018.6183,31.8997,0.2265,1.5908,4.1293,1.3760
lr,Linear Regression,21.2586,1026.6510,32.0195,0.2208,1.6469,4.1380,0.8210
ridge,Ridge Regression,21.2585,1026.6492,32.0195,0.2208,1.6469,4.1380,0.0500
br,Bayesian Ridge,21.2558,1026.6812,32.0207,0.2207,1.6486,4.1257,0.0630
en,Elastic Net,21.2618,1027.1868,32.0286,0.2203,1.6482,4.1236,0.1330
lasso,Lasso Regression,21.2729,1027.5580,32.0343,0.2200,1.6493,4.1235,0.6160
llar,Lasso Least Angle Regression,21.2730,1027.5638,32.0343,0.2200,1.6493,4.1235,0.0530
lightgbm,Light Gradient Boosting Machine,20.9904,1033.4171,32.1285,0.2157,1.5704,4.1343,0.4310
rf,Random Forest Regressor,22.2752,1070.1443,32.6989,0.1873,1.6578,4.7415,9.3710
et,Extra Trees Regressor,22.3254,1077.7063,32.8128,0.1819,1.6629,4.7747,3.1410


In [18]:
print(best_model)

GradientBoostingRegressor(random_state=42)


In [19]:
# Evaluate the model
evaluate_model(best_model)

interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

# Bowling PyCaret Model

In [24]:
reg_setup = setup(data=df_bowl_train,
                  target='bowling_fp',
                  train_size=0.99,
                  test_data = df_bowl_val,
                  data_split_shuffle=False,
                  numeric_imputation=0,
                  session_id = 42)

,Description,Value
0,Session id,42
1,Target,bowling_fp
2,Target type,Regression
3,Original data shape,"(24664, 32)"
4,Transformed data shape,"(24664, 32)"
5,Transformed train set shape,"(21208, 32)"
6,Transformed test set shape,"(3456, 32)"
7,Numeric features,31
8,Rows with missing values,35.8%
9,Preprocess,True


In [25]:
best_model = compare_models()

,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,02:16:14
Status,. . . . . . . . . . . . . . . . . .,Loading Dependencies
Estimator,. . . . . . . . . . . . . . . . . .,Compiling Library


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
br,Bayesian Ridge,14.3679,544.9260,23.3200,0.3287,1.3472,3.1934,0.0600
lr,Linear Regression,14.3626,545.0390,23.3224,0.3286,1.3457,3.1943,0.8070
ridge,Ridge Regression,14.3627,545.0382,23.3224,0.3286,1.3457,3.1943,0.0370
lasso,Lasso Regression,14.4449,546.5695,23.3548,0.3267,1.3625,3.1985,0.3900
llar,Lasso Least Angle Regression,14.4449,546.5700,23.3548,0.3267,1.3625,3.1985,0.0390
gbr,Gradient Boosting Regressor,14.3555,548.3397,23.3918,0.3245,1.3366,3.1797,1.2920
en,Elastic Net,14.6404,548.9891,23.4068,0.3237,1.4023,3.1810,0.1420
lightgbm,Light Gradient Boosting Machine,14.5037,556.7922,23.5710,0.3141,1.3333,3.1948,0.9830
rf,Random Forest Regressor,15.0936,574.4936,23.9490,0.2918,1.3884,3.5343,5.1070
et,Extra Trees Regressor,15.0895,575.8196,23.9765,0.2902,1.3883,3.5236,3.1810


In [26]:
print(best_model)

BayesianRidge()


In [27]:
# Evaluate the model
evaluate_model(best_model)

interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

In [28]:
print("Done")

Done
